In [ ]:
#analyzing congress right now 
import pandas as pd
import plotly.io as pio
import plotly.express as px
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

pio.renderers.default = "browser"

# reading
votes = pd.read_csv('data/HS119/HS119_votes.csv')
members = pd.read_csv('data/HS119/HS119_members.csv')
roll_calls = pd.read_csv('data/HS119   /HS119_rollcalls.csv')

house_votes = votes[votes["chamber"] == "House"]
senate_votes = votes[votes["chamber"] == "Senate"]

house_members = members[members["chamber"] == "House"]
senate_members = members[members["chamber"] == "Senate"]

# mapping
def convert_vote(code):
    if code in [1, 2, 3]:
        return 1;
    elif code in [4, 5, 6]:
        return -1;
    else:
        return 0;

house_votes["votes_numeric"] = votes["cast_code"].apply(convert_vote)
senate_votes["votes_numeric"] = votes["cast_code"].apply(convert_vote)

house_matrix = house_votes.pivot_table(index="icpsr", columns="rollnumber", values="votes_numeric", fill_value=0)
senate_matrix = senate_votes.pivot_table(index="icpsr", columns="rollnumber", values="votes_numeric", fill_value=0)

# Keep only legislators with at least 100 cast votes (non-zero entries).
house_matrix = house_matrix[(house_matrix != 0).sum(axis=1) >= 100]
senate_matrix = senate_matrix[(senate_matrix != 0).sum(axis=1) >= 100]


# L2-normalize each legislator vector (row) before PCA
house_matrix_norm = pd.DataFrame(
    normalize(house_matrix, norm="l2", axis=1),
    index=house_matrix.index,
    columns=house_matrix.columns,
 )
senate_matrix_norm = pd.DataFrame(
    normalize(senate_matrix, norm="l2", axis=1),
    index=senate_matrix.index,
    columns=senate_matrix.columns,
 )

# PCA

pca = PCA(n_components=2)
coords_house = pca.fit_transform(house_matrix_norm)
coords_senate = pca.fit_transform(senate_matrix_norm)

# plot house
df_house = pd.DataFrame(coords_house, columns=["PC1", "PC2"])
df_house["icpsr"] = house_matrix_norm.index

df_house = df_house.merge(
    house_members[["icpsr", "party_code", "bioname"]],
    on="icpsr"
)


fig = px.scatter(
    df_house,
    title="House",
    x="PC1",
    y="PC2",
    color="party_code",
    hover_name="bioname"
)

fig.show()

# plot senate
df_senate = pd.DataFrame(coords_senate, columns=["PC1", "PC2"])
df_senate["icpsr"] = senate_matrix_norm.index

df_senate = df_senate.merge(
    senate_members[["icpsr", "party_code", "bioname"]],
    on="icpsr"
)


fig = px.scatter(
    df_senate,
    title="Senate",
    x="PC1",
    y="PC2",
    color="party_code",
    hover_name="bioname"
)

fig.show()

In [1]:
# all congress polarization
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
import subprocess
from pathlib import Path

# reading
votes = pd.read_csv('data/all/HSall_votes.csv')
members = pd.read_csv('data/all/HSall_members.csv')

def convert_vote(code):
    if code in [1, 2, 3]:
        return 1
    elif code in [4, 5, 6]:
        return -1
    elif code in [7, 8]:
        return 0
    else:
        return np.nan
    
votes["vote_numeric"] = votes["cast_code"].apply(convert_vote)

polarization_results = []

for congress in sorted(votes["congress"].unique()):
    v = votes[votes["congress"] == congress]

    matrix = v.pivot_table(
        index="icpsr",
        columns="rollnumber",
        values="vote_numeric"
    )

    # Require at least 100 votes
    vote_counts = matrix.notna().sum(axis=1)
    matrix = matrix[vote_counts >= 100]

    if matrix.shape[0] < 10 or matrix.shape[1] < 5:
        continue

    M = matrix.fillna(0).values

    # Row normalize
    norms = np.linalg.norm(M, axis=1, keepdims=True)
    M_norm = M / np.where(norms == 0, 1, norms)

    pca = PCA(n_components=3)
    coords = pca.fit_transform(M_norm)

    pc1_variance = np.var(coords[:, 0])
    explained_pc1 = pca.explained_variance_ratio_[0]

    polarization_results.append({
        "congress": congress,
        "pc1_variance_polarization": pc1_variance,
        "pc1_explained_variance": explained_pc1,
        "num_legislators": matrix.shape[0],
        "num_votes": matrix.shape[1]
    })

polarization_df = pd.DataFrame(polarization_results)
print(polarization_df.head())

import plotly.express as px

fig = px.line(
    polarization_df,
    x="congress",
    y="pc1_variance_polarization",
    markers=True,
    title="Party-Independent Polarization Over Time",
    labels={
        "congress": "Congress",
        "pc1_variance_polarization": "PC1 Variance"
    }
)

output_path = Path("polarization.html")
fig.write_html(output_path, include_plotlyjs="cdn", auto_open=False)
subprocess.Popen(["open", "-a", "Google Chrome", str(output_path)])




   congress  pc1_variance_polarization  pc1_explained_variance  \
0         1                   0.243826                0.277607   
1         5                   0.422047                0.448816   
2         6                   0.530714                0.608192   
3         7                   0.448568                0.523501   
4         8                   0.252008                0.283633   

   num_legislators  num_votes  
0               30        109  
1              114        202  
2               19        120  
3               73        142  
4              110        150  


<Popen: returncode: None args: ['open', '-a', 'Google Chrome', 'polarization...>

In [2]:
# modern polarization, starting in the 1970s. difference between party centroids 
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

MODERN_START = 93   # change this to 91, 92, 93, etc.
MIN_VOTES = 100

def convert_vote(code):
    if code in [1, 2, 3]:
        return 1
    elif code in [4, 5, 6]:
        return -1
    elif code in [7, 8]:
        return 0
    else:
        return np.nan

votes["vote_numeric"] = votes["cast_code"].apply(convert_vote)

modern_results = []

for congress in sorted(votes["congress"].unique()):
    if congress < MODERN_START:
        continue

    v = votes[votes["congress"] == congress]

    matrix = v.pivot_table(
        index="icpsr",
        columns="rollnumber",
        values="vote_numeric"
    )

    vote_counts = matrix.notna().sum(axis=1)
    matrix = matrix[vote_counts >= MIN_VOTES]

    if matrix.shape[0] < 10 or matrix.shape[1] < 5:
        continue

    M = matrix.fillna(0).values

    norms = np.linalg.norm(M, axis=1, keepdims=True)
    M_norm = M / np.where(norms == 0, 1, norms)

    pca = PCA(n_components=2)
    coords = pca.fit_transform(M_norm)

    df = pd.DataFrame(coords, columns=["PC1", "PC2"])
    df["icpsr"] = matrix.index

    df = df.merge(
        members[["icpsr", "party_code", "bioname"]],
        on="icpsr",
        how="left"
    )

    # Voteview: 100 = Democrat, 200 = Republican
    df = df[df["party_code"].isin([100, 200])]

    if df["party_code"].nunique() < 2:
        continue

    dem_center = df[df["party_code"] == 100][["PC1", "PC2"]].mean()
    rep_center = df[df["party_code"] == 200][["PC1", "PC2"]].mean()

    distance = np.linalg.norm(rep_center - dem_center)

    modern_results.append({
        "congress": congress,
        "party_centroid_distance": distance,
        "dem_pc1_mean": dem_center["PC1"],
        "rep_pc1_mean": rep_center["PC1"],
        "dem_pc2_mean": dem_center["PC2"],
        "rep_pc2_mean": rep_center["PC2"],
        "num_legislators": len(df),
        "num_votes": matrix.shape[1]
    })

modern_polarization = pd.DataFrame(modern_results)
print(modern_polarization.head())

import plotly.express as px

fig = px.line(
    modern_polarization,
    x="congress",
    y="party_centroid_distance",
    markers=True,
    title=f"Democrat-Republican Polarization Since Congress {MODERN_START}",
    labels={
        "congress": "Congress",
        "party_centroid_distance": "Distance Between Party Centers"
    }
)

fig.show()

   congress  party_centroid_distance  dem_pc1_mean  rep_pc1_mean  \
0        93                 0.458910     -0.197581      0.258244   
1        94                 0.536054     -0.175470      0.358367   
2        95                 0.512466     -0.177585      0.332187   
3        96                 0.612728     -0.243457      0.369038   
4        97                 0.571110     -0.275686      0.285594   

   dem_pc2_mean  rep_pc2_mean  num_legislators  num_votes  
0      0.047306     -0.005816             5469       1138  
1      0.033744     -0.014965             5541       1311  
2      0.038648     -0.013829             5492       1540  
3      0.034883      0.017985             5493       1276  
4     -0.024321      0.081184             5495        966  
